# LeetCode 26 — Remove Duplicates from Sorted Array | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Two Pointers · in-place · sorted array  
**Bài gốc:** https://leetcode.com/problems/remove-duplicates-from-sorted-array/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def removeDuplicates(self, nums: list[int]) -> int:
        ...
```

**Sau bài học, bạn có thể:**
- Phân biệt giá trị k trả về với các phần tử hợp lệ ở đầu nums.
- Giải thích được vì sao mảng đã sort cho phép phát hiện trùng ở hai số kề nhau.
- Dùng con trỏ đọc/ghi để giải O(n) thời gian, O(1) bộ nhớ phụ.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Bài toán thực sự yêu cầu gì?

Cho `nums` **đã sắp xếp không giảm**. Hãy loại bỏ giá trị trùng **ngay trên mảng `nums`** sao cho mỗi giá trị chỉ còn một lần ở **phần đầu**. Trả số phần tử duy nhất `k`.

Ví dụ:

```text
nums ban đầu: [0,0,1,1,1,2,2,3,3,4]
Sau xử lý:    [0,1,2,3,4,?, ?,?, ?,?]
return k = 5
```

Các ô `?` sau `k` **không quan trọng**: LeetCode chỉ chấm `nums[:k]`. **Không** cần xóa phần đuôi hoặc làm `len(nums)` đổi. Bài yêu cầu O(1) bộ nhớ bổ sung.

Mảng **đã sorted** nên mọi bản sao cùng giá trị nằm liền nhau. Nếu bỏ điều kiện này, thuật toán kề nhau sẽ không đúng.

In [ ]:
nums = [0,0,1,1,1,2,2,3,3,4]
print("Đầu vào:", nums)
print("Kết quả kỳ vọng ở phần đầu:", [0, 1, 2, 3, 4])

## 2. Python cơ sở — list, index, len và ghi đè

`nums[i]` đọc phần tử vị trí `i` (index từ 0); `nums[i] = value` **ghi đè** phần tử ngay tại vị trí đó. `.append` làm list dài ra, còn ghi đè thì **không thay đổi chiều dài**.

Ta cần giữ `nums` tại chỗ (**in-place**) vì chấm bài sẽ nhìn vào chính list được truyền cho hàm.

In [ ]:
demo = [1, 1, 2, 3]
print("Trước:", demo, "len =", len(demo))
demo[1] = 7
print("Ghi đè:", demo, "len =", len(demo))
print("Hai phần tử đầu:", demo[:2])

## 3. Trực giác: giữ một bản của từng giá trị

Nếu được tạo một mảng mới, ta có thể làm `result = sorted(set(nums))`. **Set** là tập không chứa phần tử trùng; `sorted` sắp xếp lại giá trị.

Nhưng `set(nums)` cần **O(n) bộ nhớ phụ** trong trường hợp có n giá trị khác nhau, nên **không đạt** yêu cầu bộ nhớ O(1) của đề. Ta chỉ dùng cách này để hiểu kết quả cần thu được và đối chiếu test.

In [ ]:
def unique_using_extra_memory(nums):
    return sorted(set(nums))

print(unique_using_extra_memory([0, 0, 1, 1, 2, 3, 3]))

## 4. Insight: vì đã sort, chỉ cần nhìn phần tử duy nhất cuối

Ta dùng **2 con trỏ**:

- `read` chạy qua từng phần tử đầu vào.
- `write` (cũng là `k`) cho biết **số phần tử duy nhất đã ghi**, tức **vị trí trống tiếp theo** để ghi.

Nếu `nums[read]` **khác** `nums[write-1]`, đây là **giá trị mới**; ghi vào `nums[write]` rồi tăng `write`. Nếu bằng nhau, bỏ qua. Ban đầu phần tử đầu là duy nhất, nên `write = 1` khi mảng không rỗng.

**Chú ý:** `nums[write-1]` là phần tử duy nhất cuối cùng, không phải `nums[read-1]` một cách máy móc.

## 5. Ví dụ tính tay

Với `[1,1,2,2,3]`:

| read | nums[read] | write trước | nums[write-1] | quyết định | phần đầu hợp lệ |
|---:|---:|---:|---:|---|---|
| 1 | 1 | 1 | 1 | trùng → bỏ qua | `[1]` |
| 2 | 2 | 1 | 1 | mới → ghi vào nums[1], write=2 | `[1,2]` |
| 3 | 2 | 2 | 2 | trùng | `[1,2]` |
| 4 | 3 | 2 | 2 | mới → nums[2]=3, write=3 | `[1,2,3]` |

Sau khi duyệt, trả `write = 3`. Các phần tử đuôi không phải output được chấm.

## 6. Nền tảng: vòng `for range(1, len(nums))`

`range(1, len(nums))` sinh các chỉ số **1 đến len(nums)-1**: bắt đầu ở vị trí 1 vì phần tử đầu tiên mặc định là giá trị duy nhất đầu tiên. Nếu `len(nums)=1`, vòng lặp không làm gì.

Với list rỗng, `nums[0]` sẽ lỗi — có thể xử lý sớm bằng `if not nums: return 0` (**đây là mở rộng tốt, không ảnh hưởng constraints gốc**).

In [ ]:
numbers = [10, 10, 20]
for read in range(1, len(numbers)):
    print("read =", read, "| giá trị =", numbers[read])
print("range của list một phần tử:", list(range(1, 1)))

## 7. Thuật toán O(n) / O(1) — ghi trực tiếp

Chúng ta không dùng list/set mới và cũng không xóa phần tử nào. Mọi thay đổi diễn ra ngay trong `nums` của đề.

In [ ]:
def deduplicate_sorted_inplace(nums: list[int]) -> int:
    if not nums:
        return 0
    write = 1
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:
            nums[write] = nums[read]
            write += 1
    return write

a = [1, 1, 2, 2, 3]
k = deduplicate_sorted_inplace(a)
print("k:", k, "| phần đầu:", a[:k], "| nums sau:", a)

## 8. Tại sao ghi đè không làm mất dữ liệu chưa đọc?

Trong mọi thời điểm `write <= read`: ta chỉ ghi ở vị trí **đã đọc hoặc trước đó**, không ghi lên phần tử ở bên phải `read` mà sẽ được đọc sau.

**Bất biến:** trước mỗi vòng, `nums[:write]` chứa **đúng các giá trị khác nhau đã thấy** theo thứ tự tăng. Nếu `nums[read]` trùng phần tử cuối thì không có gì mới; nếu khác thì ghi ở `write`, kết quả vẫn tăng. Điều này giải thích cả **tính đúng** lẫn **tính an toàn của thao tác in-place**.

## 9. Trace chạy được — read, write, mảng sau cập nhật

Trace in cả trước và sau khi thay đổi. Hãy nhìn thật kỹ tại `read=2`: `nums[write]` bị ghi đè nhưng ta **không xóa** mảng.

In [ ]:
def trace_remove_duplicates(nums):
    nums = nums.copy()  # chỉ ở bản trace, để không thay đổi list người học đưa vào
    if not nums:
        return 0
    write = 1
    print("Ban đầu:", nums, "| k=", write)
    for read in range(1, len(nums)):
        x = nums[read]; last = nums[write - 1]
        print(f"read={read}, x={x}, write={write}, last={last}", end=" ")
        if x != last:
            nums[write] = x; write += 1; print("→ mới, GHI |", nums[:write])
        else:
            print("→ trùng, BỎ QUA |", nums[:write])
    print("Kết thúc k=", write, "prefix=", nums[:write])
    return write

assert trace_remove_duplicates([0, 0, 1, 1, 2, 2, 3]) == 4

## 10. Code đúng format LeetCode

LeetCode gọi `Solution().removeDuplicates(nums)` và chỉ dùng **giá trị trả về `k` cùng với `nums[:k]`** để đánh giá. Không trả cả list, không gọi `print` trong bản submit.

In [ ]:
class Solution:
    def removeDuplicates(self, nums: list[int]) -> int:
        if not nums:
            return 0
        k = 1
        for read in range(1, len(nums)):
            if nums[read] != nums[k - 1]:
                nums[k] = nums[read]
                k += 1
        return k

nums = [1, 1, 2]
k = Solution().removeDuplicates(nums)
print("return:", k, "| nums[:k]:", nums[:k])

## 11. Test phải kiểm tra hai điều kiện

**Chỉ assert k không đủ**: hàm trả đúng số 3 nhưng ghi sai `nums[:3]` vẫn là sai. Luôn kiểm tra cả `k` và phần đầu của list. Kiểm tra thêm rằng list **không bị rút ngắn**, vì đề yêu cầu ghi tại chỗ.

In [ ]:
tests_26 = [
    ([1, 1, 2], [1, 2]),
    ([0,0,1,1,1,2,2,3,3,4], [0,1,2,3,4]),
    ([1], [1]), ([1, 1, 1], [1]),
    ([1, 2, 3], [1, 2, 3]),
    ([-3, -3, -1, 0, 0, 2], [-3, -1, 0, 2]),
    ([], []),
]
for original, expected in tests_26:
    nums = original.copy(); old_len = len(nums)
    k = Solution().removeDuplicates(nums)
    assert k == len(expected), (original, k)
    assert nums[:k] == expected, (original, nums[:k], expected)
    assert len(nums) == old_len
    assert expected == unique_using_extra_memory(original)
print("PASS", len(tests_26), "ca kiểm thử k + nums[:k] + len + đối chiếu")

## 12. Phân tích độ phức tạp

Gọi `n = len(nums)`. `read` tăng từ 1 đến n-1, mỗi vòng chỉ thực hiện so sánh, ghi và tăng con trỏ với O(1) công việc ⇒ **O(n)** thời gian.

Ngoài `write` và `read` ta không tạo cấu trúc theo n ⇒ **O(1)** bộ nhớ phụ. **Không** tính bản `trace` có `.copy()` vào lời giải, vì `.copy()` chỉ dùng để minh họa.

Cách `sorted(set(nums))` dễ hiểu nhưng tốn **O(n)** bộ nhớ phụ nên không được dùng làm lời giải tối ưu in-place.

## 13. Lỗi thường gặp

1. `return nums[:k]`: sai **kiểu output** (cần số nguyên `k`).
2. `nums = sorted(set(nums))`: chỉ **gán lại tên biến** vào list mới; list gốc bên ngoài **không đổi**.
3. `write = 0` rồi đọc `nums[write-1]`: vị trí -1 là **phần tử cuối** list, không phải “giá trị duy nhất cuối”.
4. Quên mảng đã sorted: với `[1,2,1]`, ta không thể dùng so 2 số kề nhau để lọc trùng toàn cục.
5. Cho rằng phải xóa đuôi: LeetCode chỉ chấm `nums[:k]`, đuôi tùy ý.

In [ ]:
original = [1, 1, 2]
inside = original
inside = sorted(set(inside))  # chỉ đổi biến inside
print("inside:", inside, "| original:", original)

## 14. Pattern — hai con trỏ đọc/ghi

Khi cần **lọc dữ liệu in-place**, giữ thứ tự của các phần tử cần giữ, hãy nghĩ **read / write**. `read` duyệt cả mảng; `write` đánh dấu nơi ghi phần tử hợp lệ tiếp theo.

Sang bài #27 (Remove Element), ta dùng gần như cùng tư duy, nhưng tiêu chí giữ phần tử là `x != val`, không phải “khác phần tử cuối đã giữ”.

## 15. Bài tập — làm theo thứ tự và chỉ xem đáp án sau

### Bài 1 — Đọc phần đầu hợp lệ — Tự làm

Với `nums=[2,4,5,7,7]`, `k=3`, trả `nums[:k]`.

**Gợi ý cấp 1 (tư duy):** Phần đầu là vị trí 0..k-1.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`nums[:k]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả trực tiếp `nums[:k]`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_26_1(nums: list[int], k: int) -> list[int]:
    # TODO: slicing
    pass

# assert exercise_26_1([2,4,5,7,7], 3) == [2,4,5]

### Bài 2 — Nhận diện số khác nhau bằng tay — Tự làm

Viết hàm trả đúng số lượng giá trị khác nhau trong `[1,1,1,3,3,5,9,9]` bằng cách giải thích trên giấy trước, sau đó code.

**Gợi ý cấp 1 (tư duy):** Nhìn từng cụm giống nhau.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Một cụm là một giá trị duy nhất.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Các cụm: 1, 3, 5, 9.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_26_2() -> int:
    # TODO: trả số giá trị phân biệt
    pass

# assert exercise_26_2() == 4

### Bài 3 — Điền thuật toán in-place — Tự làm

Hoàn thiện con trỏ `write=1` và vòng `read` để chỉnh list ngay tại chỗ.

**Gợi ý cấp 1 (tư duy):** Chỉ giữ số khác số duy nhất cuối.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng `range(1,len(nums))` và `nums[write-1]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`if nums[read] != nums[write-1]: nums[write] = nums[read]; write += 1`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_26_3(nums: list[int]) -> int:
    if not nums:
        return 0
    write = 1
    # TODO: for read in range(...):
    # TODO: if khác nums[write-1], ghi nums[write] và tăng write
    pass

# a = [1,1,2,2,3]
# assert exercise_26_3(a) == 3 and a[:3] == [1,2,3]

### Bài 4 — Đếm thay vì ghi — Tự làm

Với mảng đã sorted, trả **số lượng** phần tử duy nhất, không cần thay đổi list.

**Gợi ý cấp 1 (tư duy):** Bắt đầu bằng 0 khi rỗng, 1 khi không rỗng.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

So `nums[i] != nums[i-1]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Lặp `i` từ 1 đến len(nums)-1, nếu khác thì tăng k.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_26_4(nums: list[int]) -> int:
    # TODO: đếm số điểm bắt đầu của cụm mới
    pass

# assert exercise_26_4([-1,-1,0,1,1]) == 3

### Bài 5 — Biến thể giữ tối đa 2 bản — Tự làm

Cho nums sorted, giữ **tối đa hai** bản của mỗi giá trị ở `nums[:k]`, ví dụ `[1,1,1,2,2,3]` → `[1,1,2,2,3]`. Đây là biến thể của #26, **không phải** yêu cầu bài gốc.

**Gợi ý cấp 1 (tư duy):** Khi k < 2, phần tử luôn được giữ.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Khi đã giữ 2 phần tử, so với `nums[k-2]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`for x in nums: if k < 2 or x != nums[k-2]: nums[k]=x; k+=1`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_26_5(nums: list[int]) -> int:
    # TODO: read/write với điều kiện so với nums[k-2]
    pass

# a = [1,1,1,2,2,3]
# assert exercise_26_5(a) == 5 and a[:5] == [1,1,2,2,3]

## 16. Đáp án tham khảo

In [ ]:
def answer_26_1(nums, k):
    return nums[:k]

def answer_26_2():
    return 4

def answer_26_3(nums):
    if not nums:
        return 0
    write = 1
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:
            nums[write] = nums[read]
            write += 1
    return write

def answer_26_4(nums):
    if not nums:
        return 0
    k = 1
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1]:
            k += 1
    return k

def answer_26_5(nums):
    k = 0
    for x in nums:
        if k < 2 or x != nums[k - 2]:
            nums[k] = x
            k += 1
    return k

assert answer_26_1([2,4,5,7,7], 3) == [2,4,5]
assert answer_26_2() == 4
u = [1,1,2,2,3]; assert answer_26_3(u) == 3 and u[:3] == [1,2,3]
assert answer_26_4([-1,-1,0,1,1]) == 3
u = [1,1,1,2,2,3]; assert answer_26_5(u) == 5 and u[:5] == [1,1,2,2,3]
print("Đáp án tham khảo: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu read/write:** `k=1` khi nonempty → đọc từ chỉ số 1 → nếu `nums[read] != nums[k-1]` thì ghi `nums[k]=nums[read]` và tăng `k` → return `k`. **O(n)** thời gian, **O(1)** nhớ phụ.

## Checklist trước khi submit

- [ ] Tôi nhớ input được sort.
- [ ] Tôi biết trả k, không phải trả list.
- [ ] Tôi chỉ coi nums[:k] là kết quả.
- [ ] Tôi hiểu con trỏ ghi không vượt quá con trỏ đọc.
- [ ] Tôi đã kiểm tra cả giá trị k và nội dung prefix.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*